# Train & Evaluate the Personnel Package Classifier

LightGBM multiclass classifier predicting the offensive personnel package (`11`, `12`, `21`, ...) given pre-snap game state, coach tendencies, and the already-decided `play_type`/`offense_formation` (this model sits downstream of both in the pipeline: play_type → formation → **personnel** → result). See `src/fb_models/modeling/plays.py` (shared feature engineering) and `src/fb_models/modeling/personnel/` (this model's feature selection + train/predict).

**Scope**: this predicts the personnel *package* (RB/TE counts, e.g. "11" = 1 RB, 1 WR... standard shorthand for 1 RB + 1 TE + 3 WR), not which specific players are on the field -- that's a depth-chart/availability lookup against `fs.teams`, not a classifier.

**Label note**: `offense_personnel` has the same season-boundary problem `offense_formation` did, but worse -- 2016-2022 use skill-position shorthand ("1 RB, 1 TE, 3 WR", ~100 unique strings/season), while 2023+ spells out the full 11-man lineup including O-line ("1 C, 2 G, 1 QB, 1 RB, 2 T, 1 TE, 3 WR", ~1,458 unique strings in 2024 alone). `modeling/plays.py`'s `_derive_personnel_package` parses just the RB/TE counts (present in both formats) and derives the standard package code, bucketing anything outside the top-8 most common packages (`11`, `12`, `21`, `13`, `22`, `01`, `10`, `02` -- ~99% of plays across 2016-2025) into `OTHER`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, log_loss

from fb_models.dataset import load_pbp_dataset, load_participation_dataset, load_games_dataset
from fb_models.modeling.plays import build_plays
from fb_models.modeling.personnel.features import PERSONNEL_FEATURE_COLS, select_personnel_features
from fb_models.modeling.personnel.train import train_personnel_classifier
from fb_models.modeling.personnel.predict import build_live_feature_row, predict_personnel_probs

In [2]:
# Season-based split, matching play_type/formation's convention: train on
# everything before TEST_SEASON, hold out TEST_SEASON entirely. Loading the
# full range up front matters here -- the expanding coach-tendency and
# previous-play features need continuous history leading into the test season.
SEASONS = list(range(2019, 2025))
TEST_SEASON = 2024

In [3]:
pbp = load_pbp_dataset(seasons=SEASONS)
participation = load_participation_dataset(seasons=SEASONS)
games = load_games_dataset()

plays = build_plays(pbp, participation, games)
plays.shape

(221042, 97)

In [4]:
train_plays = plays[plays["season"] < TEST_SEASON]
test_plays = plays[plays["season"] == TEST_SEASON]

train_df = select_personnel_features(train_plays)
test_df = select_personnel_features(test_plays)

print(f"train: {len(train_df):,} plays (seasons {SEASONS[0]}-{TEST_SEASON - 1})")
print(f"test:  {len(test_df):,} plays (season {TEST_SEASON})")
print()
print("label distribution (train):")
print(train_df["offense_personnel_package"].value_counts(normalize=True))

train: 168,377 plays (seasons 2019-2023)
test:  33,812 plays (season 2024)

label distribution (train):
offense_personnel_package
11       0.618125
12       0.207724
21       0.066143
13       0.038438
22       0.028787
10       0.018132
OTHER    0.011177
01       0.007109
02       0.004365
Name: proportion, dtype: float64


## `class_weight="balanced"` check

Formation's own experience was that balancing against a genuinely noisy minority class (PISTOL) hurt accuracy, log_loss, *and* calibration. Testing the same question here rather than assuming either way.

In [5]:
import lightgbm as lgb

X_train, y_train = train_df[PERSONNEL_FEATURE_COLS], train_df["offense_personnel_package"]
X_test, y_test = test_df[PERSONNEL_FEATURE_COLS], test_df["offense_personnel_package"]
categorical_cols = [c for c in PERSONNEL_FEATURE_COLS if train_df[c].dtype.name == "category"]

for weight in [None, "balanced"]:
    clf_ = lgb.LGBMClassifier(
        objective="multiclass", num_class=9, n_estimators=300,
        learning_rate=0.05, num_leaves=31, random_state=42,
        verbosity=-1, class_weight=weight,
    )
    clf_.fit(X_train, y_train, categorical_feature=categorical_cols)
    y_pred_ = clf_.predict(X_test)
    y_proba_ = clf_.predict_proba(X_test)
    acc_ = (y_pred_ == y_test.to_numpy()).mean()
    loss_ = log_loss(y_test, y_proba_, labels=clf_.classes_)
    print(f"class_weight={weight}: accuracy={acc_:.3f}, log_loss={loss_:.3f}")

class_weight=None: accuracy=0.681, log_loss=0.872


class_weight=balanced: accuracy=0.547, log_loss=1.225


`class_weight="balanced"` clearly loses on both accuracy and log_loss (and, as expected, over-predicts every minority package relative to its true rate). Same pattern as formation's PISTOL: `11` personnel is a genuinely dominant, not just imbalanced, package -- `train_personnel_classifier` does not use `class_weight="balanced"`.

In [6]:
clf = train_personnel_classifier(train_df)
clf.classes_

array(['01', '02', '10', '11', '12', '13', '21', '22', 'OTHER'],
      dtype=object)

## Evaluation on the held-out season

In [7]:
X_test = test_df[PERSONNEL_FEATURE_COLS]
y_test = test_df["offense_personnel_package"]

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)

accuracy = (y_pred == y_test.to_numpy()).mean()
loss = log_loss(y_test, y_proba, labels=clf.classes_)

print(f"accuracy: {accuracy:.3f}")
print(f"log_loss: {loss:.3f}")
print()
print(classification_report(y_test, y_pred, labels=clf.classes_, zero_division=0))

accuracy: 0.681
log_loss: 0.872

              precision    recall  f1-score   support

          01       1.00      0.00      0.00       465
          02       0.09      0.01      0.01       138
          10       0.00      0.00      0.00       278
          11       0.71      0.93      0.81     22226
          12       0.51      0.27      0.35      8521
          13       0.54      0.07      0.12      1362
          21       0.00      0.00      0.00       579
          22       0.16      0.17      0.17       122
       OTHER       0.00      0.00      0.00       121

    accuracy                           0.68     33812
   macro avg       0.33      0.16      0.16     33812
weighted avg       0.63      0.68      0.62     33812



In [8]:
labels = list(clf.classes_)
cm = confusion_matrix(y_test, y_pred, labels=labels)
pd.DataFrame(cm, index=labels, columns=labels)

,01,02,10,11,12,13,21,22,OTHER
01,1,0,0,456,8,0,0,0,0
02,0,1,0,132,5,0,0,0,0
10,0,1,0,255,22,0,0,0,0
11,0,9,0,20600,1560,16,0,41,0
12,0,0,0,6101,2314,48,0,55,3
13,0,0,0,742,515,92,1,11,1
21,0,0,0,501,77,0,0,1,0
22,0,0,0,51,39,11,0,21,0
OTHER,0,0,0,106,12,3,0,0,0


### Calibration sanity check

Same rationale as play_type/formation: the simulation samples from the predicted distribution rather than taking the argmax, so mean predicted probabilities should track real-world marginal rates.

In [9]:
mean_predicted_probs = pd.Series(y_proba.mean(axis=0), index=labels, name="mean_predicted_prob")
actual_rates = y_test.value_counts(normalize=True).reindex(labels).rename("actual_rate")

pd.concat([mean_predicted_probs, actual_rates], axis=1)

,mean_predicted_prob,actual_rate
01,0.007335,0.013753
02,0.002471,0.004081
10,0.009172,0.008222
11,0.696251,0.657341
12,0.218725,0.252011
13,0.032707,0.040282
21,0.019989,0.017124
22,0.009556,0.003608
OTHER,0.003795,0.003579


## Example predictions

`predict_personnel_probs` needs `play_type` and `offense_formation` as inputs (the pipeline order is play_type → formation → personnel) plus a coach-tendency snapshot, same shape as formation's `build_live_feature_row`.

In [10]:
situational_cols = [
    "down", "ydstogo", "yardline_100", "goal_to_go", "score_differential",
    "play_type", "offense_formation",
]

scenarios = {
    "1st & 10, midfield, tied, run, shotgun": [1, 10, 50, 0, 0, "run", "SHOTGUN"],
    "3rd & long, down 3, pass, shotgun": [3, 12, 60, 0, -3, "pass", "SHOTGUN"],
    "Goal to go, 2nd down, run, under center": [2, 3, 3, 1, 3, "run", "UNDER CENTER"],
    "4th & short, run, under center": [4, 1, 40, 0, -2, "run", "UNDER CENTER"],
}

rows = []
for name, values in scenarios.items():
    sample_play = test_df.iloc[[0]].copy()
    sample_play.loc[:, situational_cols] = values
    sample_play["play_type"] = sample_play["play_type"].astype("category")
    sample_play["offense_formation"] = sample_play["offense_formation"].astype("category")

    probs = predict_personnel_probs(clf, sample_play)
    rows.append({"scenario": name, **{f"prob_{k}": v for k, v in probs.items()}})

sample_plays = pd.DataFrame(rows).set_index("scenario")
sample_plays

,prob_01,prob_02,prob_10,prob_11,prob_12,prob_13,prob_21,prob_22,prob_OTHER
scenario,,,,,,,,,
"1st & 10, midfield, tied, run, shotgun",0.000442,0.000054,0.007352,0.917147,0.070948,0.001802,0.001690,0.000148,0.000417
"3rd & long, down 3, pass, shotgun",0.002318,0.000111,0.007384,0.977846,0.010409,0.000985,0.000706,0.000043,0.000196
"Goal to go, 2nd down, run, under center",0.000067,0.000048,0.002658,0.834890,0.148861,0.010711,0.001753,0.000266,0.000745
"4th & short, run, under center",0.000330,0.000018,0.004887,0.857067,0.124303,0.010284,0.002065,0.000664,0.000382


## Save artifact

Retrains on the full dataset (no held-out season) before saving.

In [11]:
import joblib

from fb_models.config import MODELS_DIR

full_df = select_personnel_features(plays)
final_clf = train_personnel_classifier(full_df)

MODELS_DIR.mkdir(exist_ok=True)

artifact_path = MODELS_DIR / "personnel_classifier.joblib"
joblib.dump(final_clf, artifact_path)
print(f"saved {artifact_path} ({len(full_df):,} plays, seasons {SEASONS[0]}-{SEASONS[-1]})")

saved /home/davis/projects/fb_models/models/personnel_classifier.joblib (202,189 plays, seasons 2019-2024)
